# Notebook 04 — Analisis Asosiasi: Market Basket Analysis

**Fase 2 · Minilab EduBI · Data Mining**

---

## Tujuan
Menemukan **pola produk yang sering dibeli bersamaan** oleh pelanggan menggunakan
algoritma **FP-Growth** (Frequent Pattern Growth).

Hasil analisis berguna untuk:
- Rekomendasi produk (*"Pelanggan yang membeli X juga membeli Y"*)
- Strategi cross-selling dan bundling produk
- Tata letak toko / urutan tampilan di platform

## Konsep Utama
| Metrik    | Definisi |
|-----------|----------|
| **Support**    | P(X ∩ Y) — seberapa sering X dan Y muncul bersamaan |
| **Confidence** | P(Y \| X) — jika beli X, seberapa sering beli Y |
| **Lift**       | Confidence / P(Y) — apakah asosiasi lebih kuat dari peluang acak |

## Alur
```
ClickHouse silver.silver_sales
    ↓
Buat transaction matrix (order_id × category)
    ↓
FP-Growth → Frequent Itemsets (min_support)
    ↓
Association Rules (min_confidence, min_lift)
    ↓
Visualisasi & interpretasi
    ↓
Log ke MLflow
```

## Referensi
- Agrawal, R. & Srikant, R. (1994). *Fast Algorithms for Mining Association Rules*. VLDB.
- Han, J., Pei, J. & Yin, Y. (2000). *Mining Frequent Patterns without Candidate Generation*. SIGMOD.
- mlxtend: https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/fpgrowth/

---
## 1. Setup & Koneksi

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import clickhouse_connect
import mlflow

from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

CH_HOST    = os.getenv('CH_HOST', 'localhost')
CH_PORT    = int(os.getenv('CH_PORT', 8123))
CH_USER    = os.getenv('CH_USER', 'default')
CH_PASS    = os.getenv('CH_PASSWORD', '')
MLFLOW_URI = os.getenv('MLFLOW_TRACKING_URI', 'http://localhost:5000')

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment('04_association_market_basket')

client = clickhouse_connect.get_client(
    host=CH_HOST, port=CH_PORT,
    username=CH_USER, password=CH_PASS
)
print('Koneksi ClickHouse berhasil.')

---
## 2. Load Data Transaksi

In [ ]:
query = """
SELECT
    order_id,
    customer_id,
    product_name,
    category,
    quantity,
    toFloat64(total_price) AS total_price,
    branch
FROM silver.silver_sales
WHERE status = 'done'
ORDER BY order_id
"""

df = client.query_df(query)
print(f'Total transaksi: {len(df)}')
print(f'Order unik    : {df["order_id"].nunique()}')
print(f'Produk unik   : {df["product_name"].nunique()}')
print(f'Kategori      : {df["category"].unique().tolist()}')
df.head()

---
## 3. Eksplorasi Distribusi Produk & Kategori

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Top 10 produk
top_products = df.groupby('product_name')['order_id'].count().sort_values(ascending=False).head(10)
top_products.plot.barh(ax=ax1, color='steelblue')
ax1.set(title='Top 10 Produk Terlaris', xlabel='Jumlah Order')
ax1.invert_yaxis()

# Distribusi kategori
cat_dist = df.groupby('category')['order_id'].count().sort_values(ascending=False)
cat_dist.plot.bar(ax=ax2, color='coral', rot=20)
ax2.set(title='Distribusi Order per Kategori', ylabel='Jumlah Order')

plt.tight_layout()
plt.savefig('experiments/mba_product_distribution.png', dpi=100)
plt.show()

---
## 4. Buat Transaction Matrix

In [ ]:
# Kelompokkan produk per order → list of lists
# Gunakan category sebagai item (lebih stabil dari product_name untuk dataset kecil)
basket = df.groupby('order_id')['category'].apply(list).reset_index()
transactions = basket['category'].tolist()

print(f'Total transaksi: {len(transactions)}')
print('Contoh transaksi:', transactions[:3])

# Encode ke format boolean matrix
te = TransactionEncoder()
te_array = te.fit_transform(transactions)
df_encoded = pd.DataFrame(te_array, columns=te.columns_)

print(f'\nDimensi matrix: {df_encoded.shape}')
df_encoded.head()

---
## 5. FP-Growth: Frequent Itemsets

In [ ]:
MIN_SUPPORT = 0.05   # item muncul di minimal 5% transaksi (sesuaikan dengan ukuran data)

frequent_itemsets = fpgrowth(
    df_encoded,
    min_support=MIN_SUPPORT,
    use_colnames=True
)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
frequent_itemsets = frequent_itemsets.sort_values('support', ascending=False)

print(f'Frequent itemsets ditemukan: {len(frequent_itemsets)}')
print('\nTop 10 frequent itemsets:')
frequent_itemsets.head(10)

---
## 6. Aturan Asosiasi (Association Rules)

In [ ]:
MIN_CONFIDENCE = 0.3
MIN_LIFT       = 1.0   # lift > 1 artinya asosiasi positif

rules = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=MIN_CONFIDENCE
)
rules = rules[rules['lift'] >= MIN_LIFT].sort_values('lift', ascending=False)

# Format tampilan
rules['antecedents'] = rules['antecedents'].apply(lambda x: ', '.join(sorted(x)))
rules['consequents'] = rules['consequents'].apply(lambda x: ', '.join(sorted(x)))

print(f'Aturan asosiasi ditemukan: {len(rules)}')
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)

In [ ]:
# Visualisasi Support vs Confidence (ukuran bubble = lift)
if len(rules) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    scatter = ax.scatter(
        rules['support'],
        rules['confidence'],
        c=rules['lift'],
        s=rules['lift'] * 80,
        cmap='YlOrRd',
        alpha=0.7,
        edgecolors='gray'
    )
    plt.colorbar(scatter, ax=ax, label='Lift')
    ax.set(xlabel='Support', ylabel='Confidence',
           title='Association Rules: Support vs Confidence (ukuran = Lift)')
    for _, row in rules.head(5).iterrows():
        ax.annotate(
            f"{row['antecedents']} → {row['consequents']}",
            (row['support'], row['confidence']),
            textcoords='offset points', xytext=(5, 5), fontsize=8
        )
    plt.tight_layout()
    plt.savefig('experiments/mba_rules_scatter.png', dpi=100)
    plt.show()
else:
    print('Tidak ada aturan asosiasi yang ditemukan. Coba kurangi min_support atau min_confidence.')

---
## 7. Log Hasil ke MLflow

In [ ]:
with mlflow.start_run(run_name='fpgrowth_category'):
    mlflow.log_param('min_support',    MIN_SUPPORT)
    mlflow.log_param('min_confidence', MIN_CONFIDENCE)
    mlflow.log_param('min_lift',       MIN_LIFT)
    mlflow.log_param('item_level',     'category')

    mlflow.log_metric('n_frequent_itemsets', len(frequent_itemsets))
    mlflow.log_metric('n_rules',             len(rules))
    if len(rules) > 0:
        mlflow.log_metric('max_lift',     rules['lift'].max())
        mlflow.log_metric('avg_confidence', rules['confidence'].mean())
        mlflow.log_artifact('experiments/mba_rules_scatter.png')

    mlflow.log_artifact('experiments/mba_product_distribution.png')

    # Simpan rules ke CSV
    rules_path = 'experiments/association_rules.csv'
    rules.to_csv(rules_path, index=False)
    mlflow.log_artifact(rules_path)

    print(f'Log berhasil. Frequent itemsets: {len(frequent_itemsets)} | Rules: {len(rules)}')

---
## 8. Interpretasi Bisnis

In [ ]:
if len(rules) > 0:
    print('=== TOP 5 REKOMENDASI CROSS-SELLING ===')
    print()
    for _, row in rules.head(5).iterrows():
        print(f'Jika pelanggan membeli : {row["antecedents"]}')
        print(f'Rekomendasikan         : {row["consequents"]}')
        print(f'Confidence: {row["confidence"]:.1%}  |  Lift: {row["lift"]:.2f}  |  Support: {row["support"]:.1%}')
        print('-' * 60)

---
## 9. Kesimpulan

**Pertanyaan Diskusi:**
1. Apa perbedaan antara algoritma Apriori dan FP-Growth? Mengapa FP-Growth lebih efisien?
2. Lift = 1 artinya apa? Kapan sebuah aturan asosiasi dianggap bermakna?
3. Jika data transaksi sangat jarang (sparse), bagaimana efeknya terhadap min_support?
4. Bagaimana aturan asosiasi yang ditemukan dapat diimplementasikan di platform e-commerce?

**Lihat hasil eksperimen di MLflow:** http://localhost:5000